In [1]:
!pip install --upgrade pip setuptools wheel
!pip install "numpy<2" "pandas<2.2" "scipy<1.11" "scikit-learn<1.4" "lightgbm==3.3.5"
!pip install pycaret==3.3.2

  Using cached scipy-1.10.1-cp39-cp39-macosx_12_0_arm64.whl.metadata (53 kB)
  Using cached scikit_learn-1.3.2-cp39-cp39-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached lightgbm-3.3.5.tar.gz (1.5 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Using cached scipy-1.10.1-cp39-cp39-macosx_12_0_arm64.whl (28.9 MB)
Using cached scikit_learn-1.3.2-cp39-cp39-macosx_12_0_arm64.whl (9.5 MB)
  error: subprocess-exited-with-error
  
  × Building wheel for lightgbm (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [104 lines of output]
      /private/var/folders/br/qhn11dt91tvcg9cqgczhjdlr0000gn/T/pip-build-env-gcjvmj6u/overlay/lib/python3.9/site-packages/setuptools/dist.py:759: SetuptoolsDeprecationWarning: License classifiers are deprecated.
      !!
      
              ********************************************************************************
              Please consider remo

In [2]:
import pandas as pd
from pycaret.regression import *
from sklearn.model_selection import train_test_split

SEED = 128
df = pd.read_csv("../Dataset2_Demand/6_Elec_Demand_Final.csv")
df_sample = df.sample(50000, random_state=SEED)

train_df, val_df = train_test_split(
    df_sample, 
    test_size=0.3, 
    random_state=SEED
)

reg = setup(
    data=train_df,
    target="england_wales_demand",
    session_id=SEED,
    fold=2,
    verbose=True
)

best_model = compare_models(sort="R2", n_select=1, turbo=True)

tuned_model = tune_model(
    best_model, 
    optimize="R2", 
    fold=5, 
    n_iter=20
)

final_model = finalize_model(tuned_model)

predictions = predict_model(final_model, data=val_df)

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def eval_preds(pred):
    y_true = pred["england_wales_demand"]
    y_pred = pred["prediction_label"]
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mean_squared_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred, squared=False),
        "R2": r2_score(y_true, y_pred)
    }

eval_final = eval_preds(predictions)
print(eval_final)

save_model(final_model, "../Model_ElecDemand/england_wales_demand_model")

,Description,Value
0,Session id,128
1,Target,england_wales_demand
2,Target type,Regression
3,Original data shape,"(35000, 6)"
4,Transformed data shape,"(35000, 6)"
5,Transformed train set shape,"(24500, 6)"
6,Transformed test set shape,"(10500, 6)"
7,Numeric features,5
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
lightgbm,Light Gradient Boosting Machine,2881.3495,14103910.8390,3755.4607,0.7288,0.1135,0.0901,0.8000
rf,Random Forest Regressor,2904.5349,14638642.7495,3825.8144,0.7185,0.1156,0.0905,0.5950
et,Extra Trees Regressor,2987.2862,15411754.5338,3925.6777,0.7037,0.1189,0.0933,0.0850
gbr,Gradient Boosting Regressor,3077.8916,15707736.5979,3963.2971,0.6980,0.1201,0.0966,0.2300
dt,Decision Tree Regressor,3232.8552,18546972.3909,4306.5345,0.6434,0.1316,0.1009,0.0200
ada,AdaBoost Regressor,3757.0805,21543676.1821,4641.5048,0.5858,0.1444,0.1213,0.4850
knn,K Neighbors Regressor,4399.4442,30145952.6803,5490.4442,0.4204,0.1729,0.1420,0.4500
llar,Lasso Least Angle Regression,5167.3592,39181868.3378,6259.5422,0.2467,0.1945,0.1666,0.0050
ridge,Ridge Regression,5167.3569,39181867.8435,6259.5421,0.2467,0.1945,0.1666,0.4150
br,Bayesian Ridge,5167.5145,39181888.9011,6259.5438,0.2467,0.1945,0.1666,0.0050


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,2756.4209,13384659.2899,3658.5051,0.7458,0.1103,0.0856
1,2781.5114,13695285.2065,3700.7141,0.7348,0.1113,0.0862
2,2803.1836,14026506.5239,3745.1978,0.7275,0.1140,0.0883
3,2798.5654,13977143.2878,3738.6018,0.7318,0.1131,0.0874
4,2840.6764,14268080.0289,3777.3112,0.7264,0.1128,0.0880
Mean,2796.0716,13870334.8674,3724.0660,0.7333,0.1123,0.0871
Std,27.6788,303487.0636,40.8324,0.0070,0.0013,0.0010


Fitting 5 folds for each of 20 candidates, totalling 100 fits


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Light Gradient Boosting Machine,2765.5045,13738749.4856,3706.5819,0.7402,0.1115,0.0859


[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3
[LightGBM] [Warning] feature_fraction is set=0.6, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6
[LightGBM] [Warning] bagging_fraction is set=1.0, subsample=1.0 will be ignored. Current value: bagging_fraction=1.0
{'MAE': 2765.504490255266, 'MSE': 13738749.485618094, 'RMSE': 3706.581914057491, 'R2': 0.7401902622622359}
Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('numerical_imputer',
                  TransformerWrapper(include=['settlement_period',
                                              'embedded_wind_generation',
                                              'embedded_wind_capacity',
                                              'embedded_solar_generation',
                                              'embedded_solar_capacity'],
                                     transformer=SimpleImputer())),
                 ('categorical_imputer',
                  TransformerWrapper(include=[],
                                     transformer=SimpleImputer(strategy='most_frequent'))),
                 ('actual_estimator',
                  LGBMRegressor(bagging_fraction=1.0, bagging_freq=3,
                                feature_fraction=0.6, learning_rate=0.2,
                                min_child_samples=1, min_split_gain=0.4,
                                n_estimators=290, n_job